In [ ]:
%load_ext autoreload
%autoreload 2

import nest_asyncio
nest_asyncio.apply()

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium',
        'figure.figsize': (18, 8),
        'axes.labelsize': 'medium',
        'axes.titlesize': 'large',
        'xtick.labelsize': 'medium',
        'ytick.labelsize': 'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import datetime
import warnings
warnings.filterwarnings('ignore')

import pytz
NYC = pytz.timezone('America/New_York')

import sys
sys.path.append('../../')

# SFR Fly RV Backtest (Event-Driven)

Uses the `QueryDrivenBacktest` framework with `SFRFlyEntryTrigger` / `SFRFlyExitTrigger`.

**Strategy:** Enter SOFR 3M microflies when z-score exceeds threshold, exit on mean-reversion, take-profit, stop-loss, or max holding.

**Tweak `BACKTEST_CONFIG` below and re-run.**

---
## 1. Configuration

In [ ]:
BACKTEST_CONFIG = {
    # -- Data --
    'data_start': '2024-06-01',
    'bt_start':   '2025-01-02',
    'bt_end':     'live',
    'n_contracts': 12,
    'constant_maturity': True,
    'roll_adjusted': True,
    'source': 'BARCHART_STIRF-RL',
    'curve': 'USD-SOFR-1D-Q12STIRT',

    # -- Structure --
    'fly_gap': 1,                      # 1=3M, 2=6M, 3=9M, 4=12M

    # -- Z-Score / Vol windows --
    'zscore_window': 60,
    'vol_window': 20,

    # -- Entry (all must be met) --
    'entry_min_zscore': 1.5,
    'entry_require_carry': False,
    'entry_min_risk_adj_roll': 0.0,
    'entry_max_vol': None,

    # -- Exit (first match wins) --
    'exit_mean_reversion': True,
    'exit_take_profit_zscore': None,   # e.g. 0.5
    'exit_take_profit_bp': None,       # e.g. 3.0
    'exit_stop_loss_sd': 2.0,
    'exit_stop_loss_bp': None,         # e.g. -5.0
    'exit_max_holding_days': 22,

    # -- Portfolio --
    'max_concurrent_trades': 3,
    'no_duplicate_flies': True,

    # -- Sizing & Costs --
    'belly_bpv': 100_000,
    'round_trip_cost_bp': 0.5,
}

C = BACKTEST_CONFIG
gap_label = {1: '3M', 2: '6M', 3: '9M', 4: '12M'}[C['fly_gap']]
print(f'Strategy: {gap_label} Fly RV | Z>{C["entry_min_zscore"]} | MaxHold={C["exit_max_holding_days"]}d')

---
## 2. Load Data & Compute Signals

In [ ]:
from BT.signals.sfr_cal_spread_rv import (
    SFRCalSpreadRVConfig, load_rate_panel, compute_fly_curve, compute_zscore_ts,
)
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.TimeseriesBuilder import TimeseriesBuilder

sfr_config = SFRCalSpreadRVConfig(
    n_contracts=C['n_contracts'], zscore_window=C['zscore_window'],
    vol_window=C['vol_window'], constant_maturity=C['constant_maturity'],
    roll_adjusted=C['roll_adjusted'], source=C['source'], curve=C['curve'],
)

curve_mdp = IRSwapsMDP(source=sfr_config.source)
ts_builder = TimeseriesBuilder()

start = NYC.localize(datetime.datetime.fromisoformat(C['data_start']).replace(hour=18))
print(f'Loading rate panel ({C["data_start"]} -> {C["bt_end"]})...')
rates = load_rate_panel(sfr_config, start=start, end=C['bt_end'],
                        curve_mdp=curve_mdp, ts_builder=ts_builder)
print(f'  {rates.shape[0]} dates x {rates.shape[1]} contracts')
print(f'  Columns: {list(rates.columns)}')
rates.tail(3)

In [ ]:
gap = C['fly_gap']
fly_ts = compute_fly_curve(rates, gap=gap)
zscore_ts = compute_zscore_ts(fly_ts, window=C['zscore_window'])
vol_ts = fly_ts.diff().rolling(C['vol_window'], min_periods=10).std() * np.sqrt(252)

roll_ts = pd.DataFrame(np.nan, index=fly_ts.index, columns=fly_ts.columns)
for i in range(len(fly_ts)):
    row = fly_ts.iloc[i]
    for j in range(1, len(fly_ts.columns)):
        roll_ts.iloc[i, j] = row.iloc[j - 1] - row.iloc[j]
risk_adj_roll_ts = roll_ts / vol_ts.replace(0, np.nan)

print(f'{gap_label} Flies: {list(fly_ts.columns)}')
print(f'First valid z-score: {zscore_ts.dropna(how="all").index[0]}')

---
## 3. Build Signal Table & Wire Triggers

In [ ]:
from BT.signals.sfr_fly_triggers import (
    build_signal_table, SFRFlyEntryTrigger, SFRFlyExitTrigger,
)

signal_table = build_signal_table(
    fly_ts, zscore_ts, vol_ts, roll_ts, risk_adj_roll_ts, C,
)

total_signals = sum(len(v) for v in signal_table.values())
passing_signals = sum(sum(1 for s in v if s.passes_entry) for v in signal_table.values())
print(f'Signal table: {len(signal_table)} dates with signals')
print(f'Total signals: {total_signals}, passing entry filters: {passing_signals}')

In [ ]:
from BT.data_handler import TimeGrid
from BT.query_engine import QueryDrivenBacktest
from BT.query_strategy import QueryStrategy

entry_trigger = SFRFlyEntryTrigger(signal_table, C)
exit_trigger = SFRFlyExitTrigger(signal_table, C)

strategy = QueryStrategy(
    name=f'sfr_{gap_label.lower()}_fly_rv',
    triggers=[entry_trigger, exit_trigger],
    default_mdp=curve_mdp,
)

print(f'Strategy: {strategy.name}')
print(f'Triggers: {len(strategy.triggers)} (entry + exit)')

---
## 4. Run Backtest

In [ ]:
bt_start_dt = NYC.localize(datetime.datetime.fromisoformat(C['bt_start']).replace(hour=17))
bt_end_dt = NYC.localize(datetime.datetime.now()) if C['bt_end'] == 'live' else \
            NYC.localize(datetime.datetime.fromisoformat(C['bt_end']).replace(hour=17))

bt_dates = pd.bdate_range(bt_start_dt, bt_end_dt, tz=NYC)
bt_datetimes = [d.to_pydatetime() for d in bt_dates]

bt = QueryDrivenBacktest(
    time_grid=TimeGrid(bt_datetimes),
    strategy=strategy,
    mdp=curve_mdp,
)

print(f'Running: {bt_start_dt.date()} to {bt_end_dt.date()} ({len(bt_datetimes)} steps)')
bt.run()

---
## 5. Results

In [ ]:
mtm = pd.Series(bt.mtm_history).sort_index()
realized = bt.realized_pnl
final_mtm = mtm.iloc[-1] if len(mtm) > 0 else 0

daily_pnl = mtm.diff().dropna()
std_d = daily_pnl.std()
sharpe = (daily_pnl.mean() / std_d * np.sqrt(252)) if std_d > 0 else 0
peak = mtm.cummax()
dd = mtm - peak
max_dd = dd.min()
hit_rate = (daily_pnl > 0).mean()
n_trades = len(bt.portfolio.trades_log)

print(f"{'=' * 60}")
print(f'{gap_label} FLY RV BACKTEST RESULTS')
print(f"{'=' * 60}")
print(f'Period:            {bt_start_dt.date()} to {bt_end_dt.date()}')
print(f'Total entries:     {n_trades}')
print(f'Open positions:    {len(bt.portfolio.positions)}')
print(f'Final MTM P&L:     ${final_mtm:,.0f}')
print(f'Realized P&L:      ${realized:,.0f}')
print(f'Sharpe ratio:      {sharpe:.2f}')
print(f'Max drawdown:      ${max_dd:,.0f}')
print(f'Daily hit rate:    {hit_rate:.1%}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 10), gridspec_kw={'height_ratios': [3, 1]})

ax = axes[0]
mtm.plot(ax=ax, linewidth=1.5, color='tab:cyan')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title(f'{gap_label} Fly RV -- Cumulative MTM P&L ($) | Sharpe={sharpe:.2f}',
             fontweight='bold')
ax.set_ylabel('P&L ($)')
ax.grid(True, alpha=0.3)

ax = axes[1]
dd.plot(ax=ax, color='red', linewidth=1)
ax.fill_between(dd.index, dd.values, 0, color='red', alpha=0.2)
ax.set_title(f'Drawdown | Max DD = ${max_dd:,.0f}', fontweight='bold')
ax.set_ylabel('Drawdown ($)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Trade Log

In [ ]:
if bt.portfolio.trades_log:
    log_df = pd.DataFrame([{
        'Date': str(getattr(o, 'timestamp', ''))[:10],
        'Action': (o.meta or {}).get('action', ''),
        'Tags': ', '.join((o.meta or {}).get('tags', [])),
        'Direction': (o.meta or {}).get('direction', ''),
        'Entry Z': (o.meta or {}).get('entry_zscore', ''),
        'Entry Lvl': (o.meta or {}).get('entry_level', ''),
    } for o in bt.portfolio.trades_log])
    print(f'Trade log ({len(log_df)} entries):')
    display(log_df)
else:
    print('No trades executed')

In [ ]:
# Closed positions with exit reasons
if hasattr(bt.portfolio, 'unwind_log') and bt.portfolio.unwind_log:
    unwind_df = pd.DataFrame([{
        'Date': str(getattr(u, 'timestamp', ''))[:10],
        'Reason': (u.meta or {}).get('reason', ''),
    } for u in bt.portfolio.unwind_log])
    print('Unwind log:')
    display(unwind_df.groupby('Reason').size().reset_index(name='Count'))
elif hasattr(bt.portfolio, 'closed_positions_log') and bt.portfolio.closed_positions_log:
    print(f'Closed positions: {len(bt.portfolio.closed_positions_log)}')
    for cp in bt.portfolio.closed_positions_log[:10]:
        print(f'  {cp}')
else:
    print('No unwind/closed position log available')

---
## 7. Position History

In [ ]:
# Number of open positions over time
pos_counts = pd.Series({dt: len(positions) for dt, positions in bt.position_history.items()}).sort_index()

fig, ax = plt.subplots(figsize=(16, 4))
pos_counts.plot(ax=ax, kind='area', color='tab:cyan', alpha=0.4)
pos_counts.plot(ax=ax, color='tab:cyan', linewidth=1)
ax.set_title('Number of Open Positions Over Time', fontweight='bold')
ax.set_ylabel('# Positions')
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Exit Strategy Comparison

Run the same entry signals through different exit configurations.

In [ ]:
exit_configs = {
    'Mean Rev Only (44d)': {
        'exit_mean_reversion': True, 'exit_stop_loss_sd': None,
        'exit_take_profit_zscore': None, 'exit_take_profit_bp': None,
        'exit_stop_loss_bp': None, 'exit_max_holding_days': 44,
    },
    'MeanRev + Stop 2sd (22d)': {
        'exit_mean_reversion': True, 'exit_stop_loss_sd': 2.0,
        'exit_take_profit_zscore': None, 'exit_take_profit_bp': None,
        'exit_stop_loss_bp': None, 'exit_max_holding_days': 22,
    },
    'TP z<0.5 + Stop 2sd (22d)': {
        'exit_mean_reversion': False, 'exit_stop_loss_sd': 2.0,
        'exit_take_profit_zscore': 0.5, 'exit_take_profit_bp': None,
        'exit_stop_loss_bp': None, 'exit_max_holding_days': 22,
    },
    'TP +5bp / SL -3bp (22d)': {
        'exit_mean_reversion': False, 'exit_stop_loss_sd': None,
        'exit_take_profit_zscore': None, 'exit_take_profit_bp': 5.0,
        'exit_stop_loss_bp': -3.0, 'exit_max_holding_days': 22,
    },
    'TP z<0.3 + Stop 1.5sd (15d)': {
        'exit_mean_reversion': False, 'exit_stop_loss_sd': 1.5,
        'exit_take_profit_zscore': 0.3, 'exit_take_profit_bp': None,
        'exit_stop_loss_bp': None, 'exit_max_holding_days': 15,
    },
}

fig, ax = plt.subplots(figsize=(18, 7))
exit_summary = []

for name, overrides in exit_configs.items():
    cfg = dict(C)
    cfg.update(overrides)

    # Rebuild signal table with new exit config
    st = build_signal_table(fly_ts, zscore_ts, vol_ts, roll_ts, risk_adj_roll_ts, cfg)
    entry_t = SFRFlyEntryTrigger(st, cfg)
    exit_t = SFRFlyExitTrigger(st, cfg)
    strat = QueryStrategy(name=name, triggers=[entry_t, exit_t], default_mdp=curve_mdp)

    bt_run = QueryDrivenBacktest(
        time_grid=TimeGrid(bt_datetimes),
        strategy=strat,
        mdp=curve_mdp,
        show_progress=False,
    )
    try:
        bt_run.run()
        m = pd.Series(bt_run.mtm_history).sort_index()
        m.plot(ax=ax, label=name, linewidth=1.5)
        d = m.diff().dropna()
        s = d.std()
        exit_summary.append({
            'Strategy': name,
            'Final MTM': f'${m.iloc[-1]:,.0f}',
            'Entries': len(bt_run.portfolio.trades_log),
            'Sharpe': round(d.mean() / s * np.sqrt(252), 2) if s > 0 else 0,
            'Max DD': f'${(m - m.cummax()).min():,.0f}',
        })
    except Exception as e:
        print(f'{name}: ERROR - {e}')

ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Exit Strategy Comparison -- Cumulative MTM P&L ($)', fontweight='bold')
ax.set_ylabel('P&L ($)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n=== Exit Strategy Summary ===')
display(pd.DataFrame(exit_summary).set_index('Strategy'))

---
## 9. Carry Filter Impact

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

for carry_on, label in [(False, 'No carry filter'), (True, 'With carry filter')]:
    cfg = dict(C)
    cfg['entry_require_carry'] = carry_on
    st = build_signal_table(fly_ts, zscore_ts, vol_ts, roll_ts, risk_adj_roll_ts, cfg)
    entry_t = SFRFlyEntryTrigger(st, cfg)
    exit_t = SFRFlyExitTrigger(st, cfg)
    strat = QueryStrategy(name=label, triggers=[entry_t, exit_t], default_mdp=curve_mdp)
    bt_run = QueryDrivenBacktest(
        time_grid=TimeGrid(bt_datetimes), strategy=strat, mdp=curve_mdp, show_progress=False,
    )
    bt_run.run()
    m = pd.Series(bt_run.mtm_history).sort_index()
    m.plot(ax=ax, label=f'{label} ({len(bt_run.portfolio.trades_log)} entries)', linewidth=1.5)
    d = m.diff().dropna()
    s = d.std()
    sharpe_v = d.mean() / s * np.sqrt(252) if s > 0 else 0
    print(f'{label}: MTM=${m.iloc[-1]:,.0f} | Entries={len(bt_run.portfolio.trades_log)} | Sharpe={sharpe_v:.2f}')

ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Carry Filter Impact', fontweight='bold')
ax.set_ylabel('P&L ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()